## Table of Contents

- [Load and Inspect Training Data](#load-and-inspect-training-data)
- [Build the Filtered Sample](#build-the-filtered-sample)

### Load and Inspect Training Data

In [2]:
import pandas as pd
from datasets import load_dataset

DATASET_NAME = "Lichess/standard-chess-games"
MIN_ELO = 1800
SAMPLE_SIZE = 100

In [3]:
games = load_dataset(
    DATASET_NAME,
    split="train",
    streaming=True,
)

games

IterableDataset({
    features: ['Event', 'Site', 'White', 'Black', 'Result', 'WhiteTitle', 'BlackTitle', 'WhiteElo', 'BlackElo', 'WhiteRatingDiff', 'BlackRatingDiff', 'UTCDate', 'UTCTime', 'ECO', 'Opening', 'Termination', 'TimeControl', 'movetext'],
    num_shards: 26138
})

In [4]:
games.features

{'Event': Value('string'),
 'Site': Value('string'),
 'White': Value('string'),
 'Black': Value('string'),
 'Result': Value('string'),
 'WhiteTitle': Value('string'),
 'BlackTitle': Value('string'),
 'WhiteElo': Value('int16'),
 'BlackElo': Value('int16'),
 'WhiteRatingDiff': Value('int16'),
 'BlackRatingDiff': Value('int16'),
 'UTCDate': Value('date32'),
 'UTCTime': Value('time32[ms]'),
 'ECO': Value('string'),
 'Opening': Value('string'),
 'Termination': Value('string'),
 'TimeControl': Value('string'),
 'movetext': Value('string')}

In [5]:
# Returns one game from the IterableDataset

game = next(iter(games))

game

{'Event': 'Rated Classical game',
 'Site': 'https://lichess.org/j1dkb5dw',
 'White': 'BFG9k',
 'Black': 'mamalak',
 'Result': '1-0',
 'WhiteTitle': None,
 'BlackTitle': None,
 'WhiteElo': 1639,
 'BlackElo': 1403,
 'WhiteRatingDiff': 5,
 'BlackRatingDiff': -8,
 'UTCDate': datetime.date(2012, 12, 31),
 'UTCTime': datetime.time(23, 1, 3),
 'ECO': 'C00',
 'Opening': 'French Defense: Normal Variation',
 'Termination': 'Normal',
 'TimeControl': '600+8',
 'movetext': '1. e4 e6 2. d4 b6 3. a3 Bb7 4. Nc3 Nh6 5. Bxh6 gxh6 6. Be2 Qg5 7. Bg4 h5 8. Nf3 Qg6 9. Nh4 Qg5 10. Bxh5 Qxh4 11. Qf3 Kd8 12. Qxf7 Nc6 13. Qe8# 1-0'}

In [6]:
game["movetext"]

'1. e4 e6 2. d4 b6 3. a3 Bb7 4. Nc3 Nh6 5. Bxh6 gxh6 6. Be2 Qg5 7. Bg4 h5 8. Nf3 Qg6 9. Nh4 Qg5 10. Bxh5 Qxh4 11. Qf3 Kd8 12. Qxf7 Nc6 13. Qe8# 1-0'

### Build the Filtered Sample

In [7]:
filtered_games = games.filter(
    lambda game: (
        game["WhiteElo"] is not None
        and game["BlackElo"] is not None
        and game["WhiteElo"] >= MIN_ELO
        and game["BlackElo"] >= MIN_ELO
        and game["Result"] in {"1-0", "0-1", "1/2-1/2"}
    )
)

# Iterate through the streamed dataset, apply my filter, and stop once 100 matching games have been obtained.

sample = list(filtered_games.take(SAMPLE_SIZE))

print(f"sample type: {type(sample)}")
print(f"sample length: {len(sample)}")
print(sample[0])  # Print the first game in the sample to inspect its structure.

sample type: <class 'list'>
sample length: 100
{'Event': 'Rated Bullet game', 'Site': 'https://lichess.org/rklpc7mk', 'White': 'Naitero_Nagasaki', 'Black': '800', 'Result': '0-1', 'WhiteTitle': None, 'BlackTitle': None, 'WhiteElo': 1824, 'BlackElo': 1973, 'WhiteRatingDiff': -6, 'BlackRatingDiff': 8, 'UTCDate': datetime.date(2012, 12, 31), 'UTCTime': datetime.time(23, 4, 57), 'ECO': 'B12', 'Opening': 'Caro-Kann Defense: Goldman Variation', 'Termination': 'Normal', 'TimeControl': '60+1', 'movetext': '1. e4 c6 2. Nc3 d5 3. Qf3 dxe4 4. Nxe4 Nd7 5. Bc4 Ngf6 6. Nxf6+ Nxf6 7. Qg3 Bf5 8. d3 Bg6 9. Ne2 e6 10. Bf4 Nh5 11. Qf3 Nxf4 12. Nxf4 Be7 13. Bxe6 fxe6 14. Nxe6 Qa5+ 15. c3 Qe5+ 16. Qe3 Qxe3+ 17. fxe3 Kd7 18. Nf4 Bd6 19. Nxg6 hxg6 20. h3 Bg3+ 21. Kd2 Raf8 22. Rhf1 Ke7 23. d4 Rxf1 24. Rxf1 Rf8 25. Rxf8 Kxf8 26. e4 Ke7 27. Ke3 g5 28. Kf3 Be1 29. Kg4 Bd2 30. Kf5 Bc1 31. Kg6 Kf8 32. e5 Bxb2 33. Kxg5 Bxc3 34. h4 Bxd4 35. h5 Bxe5 36. g4 Bb2 37. Kf5 Kf7 38. g5 Bc1 39. g6+ Ke7 40. Ke5 b5 41. Kd4 Kd6

In [10]:
from dataset import iter_training_examples

# Extract training examples from a single game record
game_record = sample[0]

# Generate a list of training examples from the game record
examples = list(iter_training_examples(game_record))

print(f"len(examples): {len(examples)}")
print(f"examples: {examples[0]}")

len(examples): 94
examples: TrainingExample(board=Board('rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1'), move=Move.from_uci('e2e4'), source_game_id='https://lichess.org/rklpc7mk', ply=1, white_elo=1824, black_elo=1973)


In [11]:
df = pd.DataFrame(sample)
df.head(2)

,Event,Site,White,Black,Result,WhiteTitle,BlackTitle,WhiteElo,BlackElo,WhiteRatingDiff,BlackRatingDiff,UTCDate,UTCTime,ECO,Opening,Termination,TimeControl,movetext
0,Rated Bullet game,https://lichess.org/rklpc7mk,Naitero_Nagasaki,800,0-1,None,None,1824,1973,-6,8,2012-12-31,23:04:57,B12,Caro-Kann Defense: Goldman Variation,Normal,60+1,1. e4 c6 2. Nc3 d5 3. Qf3 dxe4 4. Nxe4 Nd7 5. ...
1,Rated Blitz game,https://lichess.org/vb3w3rmn,nichiren1967,chinokoli,1/2-1/2,None,None,1878,1940,2,-2,2012-12-31,23:04:28,B21,Sicilian Defense: McDonnell Attack,Normal,300+0,1. e4 c5 2. f4 d5 3. exd5 Qxd5 4. Nc3 Qd8 5. B...


In [12]:
import io

import chess.pgn


def count_plies(movetext: str) -> int | None:
    pgn_text = f"""
[Event "?"]
[Site "?"]
[Date "????.??.??"]
[Round "?"]
[White "?"]
[Black "?"]
[Result "*"]

{movetext}
"""

    game = chess.pgn.read_game(io.StringIO(pgn_text))

    if game is None:
        return None

    return sum(1 for _ in game.mainline_moves())


df["ply_count"] = df["movetext"].apply(count_plies)

df.iloc[0]["ply_count"]

np.int64(94)

In [13]:
assert len(examples) == df.iloc[0]["ply_count"]

In [14]:
for example in examples[:4]:
    print(
        example.ply,
        example.move.uci(),
        example.board.fen(),
    )

1 e2e4 rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1
2 c7c6 rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR b KQkq - 0 1
3 b1c3 rnbqkbnr/pp1ppppp/2p5/8/4P3/8/PPPP1PPP/RNBQKBNR w KQkq - 0 2
4 d7d5 rnbqkbnr/pp1ppppp/2p5/8/4P3/2N5/PPPP1PPP/R1BQKBNR b KQkq - 1 2


In [15]:
assert all(example.move in example.board.legal_moves for example in examples)

### Validate Sample Data

In [16]:
### Validate Sample Data
all_examples = []

# Generate all training examples from the 100-game sample.
# Each game_record represents one source game.

for game_record in sample:
    # Generate supervised training examples for one game.
    game_examples = list(iter_training_examples(game_record))

    # Validate that each source ply produced one training example.
    assert len(game_examples) == count_plies(game_record["movetext"])

    # Validate that every target move is legal from its stored board position.
    assert all(example.move in example.board.legal_moves for example in game_examples)

    # Validate contiguous 1-based ply numbering within each game.
    assert [example.ply for example in game_examples] == list(
        range(1, len(game_examples) + 1)
    )

    all_examples.extend(game_examples)

len(all_examples)

7234

In [17]:
# Validate that the total number of generated training examples
# matches the total source ply count.
assert len(all_examples) == int(df["ply_count"].sum())

# Validate that all training examples use 1-based ply numbering.
assert all(example.ply >= 1 for example in all_examples)

# Validate that source Elo metadata satisfies the exploration filter.
assert all(
    example.white_elo >= 1800 and example.black_elo >= 1800 for example in all_examples
)

In [18]:
for example in all_examples[:10]:
    print(
        example.source_game_id,
        example.ply,
        example.move.uci(),
        example.white_elo,
        example.black_elo,
    )

print("\n")
print(f"Games validated: {len(sample):,}")
print(f"Training examples: {len(all_examples):,}")
print(f"Average examples/game: {len(all_examples) / len(sample):.2f}")

https://lichess.org/rklpc7mk 1 e2e4 1824 1973
https://lichess.org/rklpc7mk 2 c7c6 1824 1973
https://lichess.org/rklpc7mk 3 b1c3 1824 1973
https://lichess.org/rklpc7mk 4 d7d5 1824 1973
https://lichess.org/rklpc7mk 5 d1f3 1824 1973
https://lichess.org/rklpc7mk 6 d5e4 1824 1973
https://lichess.org/rklpc7mk 7 c3e4 1824 1973
https://lichess.org/rklpc7mk 8 b8d7 1824 1973
https://lichess.org/rklpc7mk 9 f1c4 1824 1973
https://lichess.org/rklpc7mk 10 g8f6 1824 1973


Games validated: 100
Training examples: 7,234
Average examples/game: 72.34


In [19]:
# Validate that each TrainingExample owns an independent board snapshot.
first_game_examples = list(iter_training_examples(sample[0]))

assert first_game_examples[0].board is not first_game_examples[1].board

# Validate that applying example N's move produces example N+1's board.
from itertools import pairwise

for current_example, next_example in pairwise(first_game_examples):
    board = current_example.board.copy(stack=False)
    board.push(current_example.move)

    assert board.fen() == next_example.board.fen()

In [20]:
# Test that malformed PGN moves are correctly rejected.
bad_record = dict(sample[0])

bad_record["movetext"] = "1. e4 e5 2. Nf3 Nc6 3. THIS_IS_NOT_A_MOVE"

malformed_rejected = False

try:
    list(iter_training_examples(bad_record))
except ValueError as exc:
    malformed_rejected = True
    print(exc)

assert malformed_rejected

AssertionError: 

In [21]:
df["Site"].head(10)

0    https://lichess.org/rklpc7mk
1    https://lichess.org/vb3w3rmn
2    https://lichess.org/iclkx584
3    https://lichess.org/v778e8mr
4    https://lichess.org/0wn9o371
5    https://lichess.org/ejv9qc24
6    https://lichess.org/46a4pymi
7    https://lichess.org/qd3mwsmg
8    https://lichess.org/ewgdih6m
9    https://lichess.org/ekvammie
Name: Site, dtype: str

In [22]:
df["Site"].nunique(), len(df)

(100, 100)

In [23]:
source_game_id = str(game_record.get("Site", ""))

In [24]:
from itertools import pairwise

# Validate that applying example N's move produces example N+1's board.
for current_example, next_example in pairwise(first_game_examples):
    board = current_example.board.copy(stack=False)
    board.push(current_example.move)

    assert board.fen() == next_example.board.fen()